In [1]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

Python: c:\Users\gangu\healthcare-clinical-nlp\.venv\Scripts\python.exe
PyTorch: 2.14.0+cpu
CUDA: False


In [3]:
from gliformer import GLiFormer

model_name = "knowledgator/gliformer-base-v1"

gliformer_model = GLiFormer.from_pretrained(
    model_name,
    load_tokenizer=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"

gliformer_model = gliformer_model.to(device).eval()

print("Device:", device)

print("Model loaded successfully")

c:\Users\gangu\healthcare-clinical-nlp\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
c:\Users\gangu\healthcare-clinical-nlp\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gangu\.cache\huggingface\hub\models--knowledgator--gliformer-base-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, s

Device: cpu
Model loaded successfully


In [4]:
text = """
The patient has diabetes and reports chest pain.
Metformin 500 mg was prescribed twice daily.
"""

labels = [
    "disease",
    "symptom",
    "medication",
    "dosage",
    "frequency"
]

entities = gliformer_model.predict_entities(
    text,
    labels
)

entities

[{'start': 17,
  'end': 25,
  'text': 'diabetes',
  'label': 'disease',
  'score': 0.9993910789489746},
 {'start': 38,
  'end': 48,
  'text': 'chest pain',
  'label': 'symptom',
  'score': 0.9999686479568481},
 {'start': 50,
  'end': 59,
  'text': 'Metformin',
  'label': 'medication',
  'score': 0.6016772985458374},
 {'start': 82,
  'end': 93,
  'text': 'twice daily',
  'label': 'frequency',
  'score': 0.9997721314430237}]

In [5]:
from pathlib import Path

note_path = Path("../data/MACCROBAT2018/15939911.txt")

clinical_text = note_path.read_text(
    encoding = "utf-8"
)

print(clinical_text[:1000])

CASE: A 28-year-old previously healthy man presented with a 6-week history of palpitations.
The symptoms occurred during rest, 2–3 times per week, lasted up to 30 minutes at a time and were associated with dyspnea.
Except for a grade 2/6 holosystolic tricuspid regurgitation murmur (best heard at the left sternal border with inspiratory accentuation), physical examination yielded unremarkable findings.
An electrocardiogram (ECG) revealed normal sinus rhythm and a Wolff– Parkinson– White pre-excitation pattern (Fig.1: Top), produced by a right-sided accessory pathway.
Transthoracic echocardiography demonstrated the presence of Ebstein's anomaly of the tricuspid valve, with apical displacement of the valve and formation of an “atrialized” right ventricle (a functional unit between the right atrium and the inlet [inflow] portion of the right ventricle) (Fig.2).
The anterior tricuspid valve leaflet was elongated (Fig.2C, arrow), whereas the septal leaflet was rudimentary (Fig.2C, arrowhead)

In [6]:
clinical_labels = [
    "disease",
    "symptom",
    "medication",
    "dosage",
    "lab value",
    "diagnostic procedure",
    "therapeutic procedure",
    "age",
    "sex"
]

In [8]:
clinical_entities = gliformer_model.predict_entities(
    clinical_text,
    clinical_labels
)

print("Number of entities:", len(clinical_entities))

for entity in clinical_entities:
    print(
        f"{entity['text']: <40}",
        f"{entity['label']:<25}",
        f"{entity['score']:.3f}"
    )

Number of entities: 11
28-year-old                              age                       0.806
palpitations                             symptom                   0.704
dyspnea                                  symptom                   0.859
physical examination                     diagnostic procedure      0.521
Transthoracic echocardiography           diagnostic procedure      0.822
Ebstein's anomaly                        disease                   0.540
Contrast echocardiography                diagnostic procedure      0.784
saline                                   medication                0.863
electrophysiologic study                 diagnostic procedure      0.833
radiofrequency ablation                  therapeutic procedure     0.975
palpitations                             symptom                   0.632


In [9]:
ann_path = Path(
    "../data/MACCROBAT2018/15939911.ann"
)

annotations = ann_path.read_text(
    encoding="utf-8"
)

print(annotations[:2000])

T1	Age 8 19	28-year-old
T2	History 20 38	previously healthy
T3	Sex 39 42	man
T4	Clinical_event 43 52	presented
E1	Clinical_event:T4 
T5	Sign_symptom 31 38	healthy
E2	Sign_symptom:T5 
T6	Duration 60 66	6-week
E3	Duration:T6 
T7	Sign_symptom 78 90	palpitations
E4	Sign_symptom:T7 
T8	Coreference 96 104	symptoms
E5	Coreference:T8 
R1	IDENTICAL Arg1:E5 Arg2:E4	
T9	Clinical_event 121 125	rest
E6	Clinical_event:T9 
R2	MODIFY Arg1:E6 Arg2:E5	
T10	Frequency 127 145	2–3 times per week
R3	MODIFY Arg1:T10 Arg2:E5	
T12	Sign_symptom 206 213	dyspnea
E8	Sign_symptom:T12 
T11	Detailed_description 154 180	up to 30 minutes at a time
R4	MODIFY Arg1:T11 Arg2:E5	
T13	Sign_symptom 261 281	regurgitation murmur
E7	Sign_symptom:T13 
T14	Biological_structure 251 260	tricuspid
T15	Detailed_description 238 250	holosystolic
T16	Lab_value 228 237	grade 2/6
R5	MODIFY Arg1:T14 Arg2:E7	
R6	MODIFY Arg1:T15 Arg2:E7	
R7	MODIFY Arg1:T16 Arg2:E7	
T17	Biological_structure 301 320	left sternal border
R8	MODIFY Arg1:T17 Arg2:E

In [10]:
ground_truth_entities = []

for line in annotations.splitlines():

    if line.startswith("T"):

        parts = line.split("\t", 2)

        entity_id = parts[0]
        entity_info = parts[1]
        entity_text = parts[2]

        entity_type = entity_info.split()[0]

        ground_truth_entities.append({
            "id": entity_id,
            "type": entity_type,
            "text": entity_text
        })

for entity in ground_truth_entities:
    print(
        f"{entity['text']:<40}",
        entity["type"]
    )

28-year-old                              Age
previously healthy                       History
man                                      Sex
presented                                Clinical_event
healthy                                  Sign_symptom
6-week                                   Duration
palpitations                             Sign_symptom
symptoms                                 Coreference
rest                                     Clinical_event
2–3 times per week                       Frequency
dyspnea                                  Sign_symptom
up to 30 minutes at a time               Detailed_description
regurgitation murmur                     Sign_symptom
tricuspid                                Biological_structure
holosystolic                             Detailed_description
grade 2/6                                Lab_value
left sternal border                      Biological_structure
inspiratory accentuation                 Detailed_description
physical examinati

In [11]:
label_mapping = {
    "disease": "Disease_disorder",
    "symptom": "Sign_symptom",
    "medication": "Medication",
    "dosage": "Dosage",
    "lab value": "Lab_value",
    "diagnostic procedure": "Diagnostic_procedure",
    "therapeutic procedure": "Therapeutic_procedure",
    "age": "Age",
    "sex": "Sex"
}

In [12]:
target_types = set(label_mapping.values())

ground_truth_target = []

for line in annotations.splitlines():

    if not line.startswith("T"):
        continue

    parts = line.split("\t", 2)

    entity_id = parts[0]
    entity_info = parts[1]
    entity_text = parts[2]

    first_space = entity_info.find(" ")

    entity_type = entity_info[:first_space]
    span_info = entity_info[first_space + 1:]

    if entity_type not in target_types:
        continue

    spans = []

    for span in span_info.split(";"):
        start, end = span.split()
        spans.append(
            (int(start), int(end))
        )

    ground_truth_target.append({
        "id": entity_id,
        "type": entity_type,
        "spans": spans,
        "text": entity_text
    })

In [13]:
print(
    "Ground-truth target entities:",
    len(ground_truth_target)
)

for entity in ground_truth_target:
    print(
        f"{entity['text']:<45}",
        f"{entity['type']:<25}",
        entity["spans"]
    )

Ground-truth target entities: 38
28-year-old                                   Age                       [(8, 19)]
man                                           Sex                       [(39, 42)]
healthy                                       Sign_symptom              [(31, 38)]
palpitations                                  Sign_symptom              [(78, 90)]
dyspnea                                       Sign_symptom              [(206, 213)]
regurgitation murmur                          Sign_symptom              [(261, 281)]
grade 2/6                                     Lab_value                 [(228, 237)]
physical examination                          Diagnostic_procedure      [(353, 373)]
electrocardiogram                             Diagnostic_procedure      [(408, 425)]
ECG                                           Diagnostic_procedure      [(427, 430)]
sinus rhythm                                  Diagnostic_procedure      [(448, 460)]
Wolff– Parkinson– White pre-excitation pa

In [14]:
ground_truth_set = set()

for entity in ground_truth_target:

    for start, end in entity["spans"]:

        ground_truth_set.add(
            (start, end, entity["type"])
        )

print("Ground-truth spans:", len(ground_truth_set))

Ground-truth spans: 38


In [15]:
prediction_set = set()

for entity in clinical_entities:

    mapped_type = label_mapping[
        entity["label"]
    ]

    prediction_set.add(
        (
            entity["start"],
            entity["end"],
            mapped_type
        )
    )

print("GLiFormer predictions:", len(prediction_set))

GLiFormer predictions: 11


In [16]:
true_positives = prediction_set & ground_truth_set

false_positives = prediction_set - ground_truth_set

false_negatives = ground_truth_set - prediction_set

print("True positives:", len(true_positives))
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

True positives: 7
False positives: 4
False negatives: 31


In [17]:
tp = len(true_positives)
fp = len(false_positives)
fn = len(false_negatives)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0

recall = tp / (tp + fn) if (tp + fn) > 0 else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

Precision: 0.6364
Recall:    0.1842
F1:        0.2857


In [18]:
def show_span(span):
    start, end, entity_type = span

    return {
        "text": clinical_text[start:end],
        "type": entity_type,
        "start": start,
        "end": end
    }


print("FALSE POSITIVES")
print("-" * 60)

for span in sorted(false_positives):
    print(show_span(span))


print("\nFALSE NEGATIVES")
print("-" * 60)

for span in sorted(false_negatives):
    print(show_span(span))

FALSE POSITIVES
------------------------------------------------------------
{'text': 'Transthoracic echocardiography', 'type': 'Diagnostic_procedure', 'start': 573, 'end': 603}
{'text': 'Contrast echocardiography', 'type': 'Diagnostic_procedure', 'start': 1002, 'end': 1027}
{'text': 'saline', 'type': 'Medication', 'start': 1034, 'end': 1040}
{'text': 'radiofrequency ablation', 'type': 'Therapeutic_procedure', 'start': 1243, 'end': 1266}

FALSE NEGATIVES
------------------------------------------------------------
{'text': 'healthy', 'type': 'Sign_symptom', 'start': 31, 'end': 38}
{'text': 'man', 'type': 'Sex', 'start': 39, 'end': 42}
{'text': 'grade 2/6', 'type': 'Lab_value', 'start': 228, 'end': 237}
{'text': 'regurgitation murmur', 'type': 'Sign_symptom', 'start': 261, 'end': 281}
{'text': 'unremarkable', 'type': 'Lab_value', 'start': 382, 'end': 394}
{'text': 'electrocardiogram', 'type': 'Diagnostic_procedure', 'start': 408, 'end': 425}
{'text': 'ECG', 'type': 'Diagnostic_procedure

In [20]:
import json

with open("../data/train_ner_data.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open("../data/val_ner_data.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)

with open("../data/test_ner_data.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)


document_id = "15939911"

train_ids = {
    str(item["document_id"])
    for item in train_data
}

val_ids = {
    str(item["document_id"])
    for item in val_data
}

test_ids = {
    str(item["document_id"])
    for item in test_data
}


if document_id in train_ids:
    print("Document is in TRAIN")

elif document_id in val_ids:
    print("Document is in VALIDATION")

elif document_id in test_ids:
    print("Document is in TEST")

else:
    print("Document not found")

Document is in TRAIN


In [21]:
validation_document = val_data[0]

validation_document_id = str(
    validation_document["document_id"]
)

print(
    "Validation document:",
    validation_document_id
)

Validation document: 19860006


In [22]:
from pathlib import Path

validation_txt = Path(
    f"../data/MACCROBAT2018/{validation_document_id}.txt"
)

validation_ann = Path(
    f"../data/MACCROBAT2018/{validation_document_id}.ann"
)

print("TXT exists:", validation_txt.exists())
print("ANN exists:", validation_ann.exists())

TXT exists: True
ANN exists: True


In [23]:
validation_text = validation_txt.read_text(
    encoding="utf-8"
)

print(validation_text[:1000])
print("\nDocument length:", len(validation_text))

A 72-year-old woman underwent RF ablation for the treatment of a metachronous solitary liver metastasis from rectal adenocarcinoma, which had been treated 18 mo before with preoperative chemoradiotherapy and anterior resection.
The lesion was located in segment VIII.
The procedure went well and the patient was initially haemodynamically stable.
One hour later her condition suddenly worsened.
She was clinically dizzy and had fatigue and pallor.
We detected hypotension and tachycardia.
Emergency blood tests showed a haemoglobin level of < 7 g/dL and markedly elevated transaminase levels.
A computed tomography examination revealed an accumulation of intraperitoneal fluid in the pelvis and two liver haematomas.
The images showed two liver lacerations, one following the path of the needle (Figure ​(Figure1A1A-​-C)C) and another, unexplained laceration leading away from the first (Figure ​(Figure2).2).
The patient had no serious coughing or hiccupping after the RF treatment, or any other com

In [24]:
validation_entities = gliformer_model.predict_entities(
    validation_text,
    clinical_labels
)

print(
    "Number of predictions:",
    len(validation_entities)
)

for entity in validation_entities:
    print(
        f"{entity['text']:<40}",
        f"{entity['label']:<25}",
        f"{entity['score']:.4f}"
    )

Number of predictions: 15
72-year-old                              age                       0.7269
RF ablation                              therapeutic procedure     0.8867
rectal adenocarcinoma                    disease                   0.9258
preoperative chemoradiotherapy           therapeutic procedure     0.6845
anterior resection                       therapeutic procedure     0.7711
dizzy                                    symptom                   0.5347
fatigue and pallor                       symptom                   0.6213
hypotension                              symptom                   0.6230
tachycardia                              symptom                   0.6230
haemoglobin level of < 7 g/dL            lab value                 0.7063
transaminase levels                      lab value                 0.4392
computed tomography examination          diagnostic procedure      0.8990
RF treatment                             therapeutic procedure     0.5103
blood transf

In [25]:
from pathlib import Path


def evaluate_gliformer_document(document_id):

    # -----------------------------
    # 1. Load clinical document
    # -----------------------------

    txt_path = Path(
        f"../data/MACCROBAT2018/{document_id}.txt"
    )

    ann_path = Path(
        f"../data/MACCROBAT2018/{document_id}.ann"
    )

    text = txt_path.read_text(encoding="utf-8")
    annotations = ann_path.read_text(encoding="utf-8")


    # -----------------------------
    # 2. Run GLiFormer
    # -----------------------------

    predictions = gliformer_model.predict_entities(
        text,
        clinical_labels
    )


    # -----------------------------
    # 3. Build prediction set
    # -----------------------------

    prediction_set = set()

    for entity in predictions:

        mapped_type = label_mapping[
            entity["label"]
        ]

        prediction_set.add(
            (
                entity["start"],
                entity["end"],
                mapped_type
            )
        )


    # -----------------------------
    # 4. Build ground-truth set
    # -----------------------------

    target_types = set(
        label_mapping.values()
    )

    ground_truth_set = set()

    for line in annotations.splitlines():

        if not line.startswith("T"):
            continue

        parts = line.split("\t", 2)

        entity_info = parts[1]

        first_space = entity_info.find(" ")

        entity_type = entity_info[:first_space]

        if entity_type not in target_types:
            continue

        span_info = entity_info[
            first_space + 1:
        ]

        # Handles discontinuous annotations
        for span in span_info.split(";"):

            start, end = map(
                int,
                span.split()
            )

            ground_truth_set.add(
                (
                    start,
                    end,
                    entity_type
                )
            )


    # -----------------------------
    # 5. Compare predictions
    # -----------------------------

    true_positives = (
        prediction_set & ground_truth_set
    )

    false_positives = (
        prediction_set - ground_truth_set
    )

    false_negatives = (
        ground_truth_set - prediction_set
    )


    # -----------------------------
    # 6. Calculate metrics
    # -----------------------------

    tp = len(true_positives)
    fp = len(false_positives)
    fn = len(false_negatives)

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )


    # -----------------------------
    # 7. Return everything
    # -----------------------------

    return {
        "document_id": document_id,

        "ground_truth": len(
            ground_truth_set
        ),

        "predictions": len(
            prediction_set
        ),

        "tp": tp,
        "fp": fp,
        "fn": fn,

        "precision": precision,
        "recall": recall,
        "f1": f1,

        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives
    }

In [26]:
validation_result = evaluate_gliformer_document(
    validation_document_id
)

print(
    "Document:",
    validation_result["document_id"]
)

print(
    "Ground truth:",
    validation_result["ground_truth"]
)

print(
    "Predictions:",
    validation_result["predictions"]
)

print(
    "TP:",
    validation_result["tp"]
)

print(
    "FP:",
    validation_result["fp"]
)

print(
    "FN:",
    validation_result["fn"]
)

print(
    f"Precision: {validation_result['precision']:.4f}"
)

print(
    f"Recall:    {validation_result['recall']:.4f}"
)

print(
    f"F1:        {validation_result['f1']:.4f}"
)

Document: 19860006
Ground truth: 37
Predictions: 15
TP: 7
FP: 8
FN: 30
Precision: 0.4667
Recall:    0.1892
F1:        0.2692


In [27]:
validation_results = []

for i, item in enumerate(val_data):

    document_id = str(item["document_id"])

    result = evaluate_gliformer_document(
        document_id
    )

    validation_results.append(result)

    print(
        f"{i + 1}/{len(val_data)}",
        document_id,
        f"F1={result['f1']:.3f}"
    )

1/30 19860006 F1=0.269
2/30 19860007 F1=0.189
3/30 20671919 F1=0.189
4/30 21923918 F1=0.198
5/30 22791498 F1=0.322
6/30 24518095 F1=0.413
7/30 24526194 F1=0.400
8/30 25210224 F1=0.095
9/30 25246819 F1=0.294
10/30 25661749 F1=0.281
11/30 25759562 F1=0.151
12/30 25934795 F1=0.279
13/30 26175648 F1=0.326
14/30 26228535 F1=0.266
15/30 26327988 F1=0.459
16/30 26361431 F1=0.330
17/30 26474553 F1=0.312
18/30 27196481 F1=0.283
19/30 27661040 F1=0.380
20/30 27683825 F1=0.184
21/30 27741115 F1=0.266
22/30 27842605 F1=0.230
23/30 27974938 F1=0.183
24/30 28103924 F1=0.333
25/30 28202865 F1=0.440
26/30 28248891 F1=0.247
27/30 28265107 F1=0.242
28/30 28321073 F1=0.228
29/30 28353569 F1=0.078
30/30 28559815 F1=0.267


In [28]:
total_tp = sum(
    result["tp"]
    for result in validation_results
)

total_fp = sum(
    result["fp"]
    for result in validation_results
)

total_fn = sum(
    result["fn"]
    for result in validation_results
)

precision = (
    total_tp / (total_tp + total_fp)
    if (total_tp + total_fp) > 0
    else 0
)

recall = (
    total_tp / (total_tp + total_fn)
    if (total_tp + total_fn) > 0
    else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

print("GLiFormer — VALIDATION SET")
print("-------------------------")
print("Documents:", len(validation_results))
print("TP:", total_tp)
print("FP:", total_fp)
print("FN:", total_fn)
print()
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Micro F1:  {f1:.4f}")

GLiFormer — VALIDATION SET
-------------------------
Documents: 30
TP: 441
FP: 445
FN: 1924

Precision: 0.4977
Recall:    0.1865
Micro F1:  0.2713


In [29]:
from collections import defaultdict

type_counts = defaultdict(
    lambda: {
        "tp": 0,
        "fp": 0,
        "fn": 0
    }
)

for result in validation_results:

    for start, end, entity_type in result["true_positives"]:
        type_counts[entity_type]["tp"] += 1

    for start, end, entity_type in result["false_positives"]:
        type_counts[entity_type]["fp"] += 1

    for start, end, entity_type in result["false_negatives"]:
        type_counts[entity_type]["fn"] += 1

In [30]:
type_results = []

for entity_type, counts in type_counts.items():

    tp = counts["tp"]
    fp = counts["fp"]
    fn = counts["fn"]

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    type_results.append({
        "entity_type": entity_type,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

In [31]:
import pandas as pd

type_results_df = pd.DataFrame(
    type_results
)

type_results_df = type_results_df.sort_values(
    "f1",
    ascending=False
)

type_results_df[
    ["precision", "recall", "f1"]
] = type_results_df[
    ["precision", "recall", "f1"]
].round(3)

type_results_df

,entity_type,tp,fp,fn,precision,recall,f1
3,Age,27,12,5,0.692,0.844,0.761
4,Medication,88,22,44,0.800,0.667,0.727
8,Dosage,25,10,21,0.714,0.543,0.617
6,Disease_disorder,79,67,156,0.541,0.336,0.415
7,Sex,6,4,21,0.600,0.222,0.324
1,Therapeutic_procedure,25,50,82,0.333,0.234,0.275
5,Lab_value,68,93,401,0.422,0.145,0.216
0,Sign_symptom,64,89,471,0.418,0.120,0.186
2,Diagnostic_procedure,59,98,723,0.376,0.075,0.126


## GLiFormer V2.0 — Zero-Shot Baseline

### Experimental Setup
- Model: `knowledgator/gliformer-base-v1`
- Dataset: MACCROBAT 2018
- Evaluation split: Validation set
- Documents: 30
- Mode: Zero-shot inference
- Evaluation: Exact character-span + entity-type matching
- Target entity types: 9

### Overall Validation Results

| Metric | Result |
|---|---:|
| True Positives | 441 |
| False Positives | 445 |
| False Negatives | 1,924 |
| Precision | 0.4977 |
| Recall | 0.1865 |
| Micro F1 | **0.2713** |

### Entity-Level Results

| Entity Type | Precision | Recall | F1 |
|---|---:|---:|---:|
| Age | 0.692 | 0.844 | **0.761** |
| Medication | 0.800 | 0.667 | **0.727** |
| Dosage | 0.714 | 0.543 | **0.617** |
| Disease_disorder | 0.541 | 0.336 | 0.415 |
| Sex | 0.600 | 0.222 | 0.324 |
| Therapeutic_procedure | 0.333 | 0.234 | 0.275 |
| Lab_value | 0.422 | 0.145 | 0.216 |
| Sign_symptom | 0.418 | 0.120 | 0.186 |
| Diagnostic_procedure | 0.376 | 0.075 | 0.126 |

### Initial Findings

GLiFormer showed strong zero-shot performance for Age, Medication, and
Dosage, but recall was low for Diagnostic_procedure, Sign_symptom, and
Lab_value.

The three lowest-recall categories accounted for approximately 83% of
all false negatives.

Several errors were caused by differences between GLiFormer's predicted
span boundaries and MACCROBAT's annotation boundaries.

### Next Experiment

V2.1 will investigate whether improved natural-language label wording
can increase zero-shot recall on the validation set.

The validation set will be used for experimentation. The held-out test
set will remain untouched until the GLiFormer configuration is frozen.